## ANN Regression Problem Statement
---

In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [2]:
Dataset = pd.read_csv('assets/Churn_Modelling.csv')
df = Dataset
data = Dataset
data.head(5)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


> Removing Unnecessary Columns 
---

In [3]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data.head(3)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1


> Encoding Categorical Column "Gender"
- Changing categorical into numerical value


In [4]:
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


> OneHotEncoding to Geography
- Three values of Geo will be convert into 0, 1, 2. ANN considers them as greater and smaller. So OHE helps by over coming it.
---

In [5]:
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo = OneHotEncoder()

geo_encoder = onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [6]:
print(geo_encoder.toarray())

print(onehot_encoder_geo.get_feature_names_out(['Geography']))

geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

[[1. 0. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 ...
 [1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]]
['Geography_France' 'Geography_Germany' 'Geography_Spain']


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


> Combining OHE with Original Dataset
---

In [7]:
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data.head(3)

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0


> Saving Encoder and Scaler
---

In [8]:
with open('models/ann-customer-churn-prediction-label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('models/ann-customer-churn-prediction-onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo, file)

> Dividing independent and dependent features
---

In [9]:
X = data.drop('Exited', axis=1)
y = data['Exited']

> Train Test Split
---

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

>Standarization
---

In [11]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

> Saving Scaler file
---

In [12]:
with open('models/ann-customer-churn-prediction-scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

## ANN Implementations

> Importing Libraries
---

In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import datetime


> Builing ANN Model
---

In [14]:
models = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  #HL1
    Dense(32, activation='relu'),                                   #HL2
    Dense(1)                                                        #O/p L
])

models.summary()

d:\CODE\04-Projects\ann-customer-churn-prediction\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

> Compile the model
- used for forward and backward propogation
---

In [15]:
models.compile(optimizer='adam', loss="mean_absolute_error", metrics=['mae'])

> Setup TensorBoard
- Used for fit log visualization
---

In [16]:
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping

log_dir = 'regressionlogs/fit/' + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

> Setup EarlyStopping
---

In [17]:
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

> Train The Model
---

In [18]:
history = models.fit(
    X_train, y_train, validation_data=(X_test,y_test), epochs=100,
    callbacks = [early_stopping_callback, tensorflow_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.2434 - mae: 0.2434 - val_loss: 0.2142 - val_mae: 0.2142
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2105 - mae: 0.2105 - val_loss: 0.1987 - val_mae: 0.1987
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1939 - mae: 0.1939 - val_loss: 0.1860 - val_mae: 0.1860
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1840 - mae: 0.1840 - val_loss: 0.1784 - val_mae: 0.1784
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1778 - mae: 0.1778 - val_loss: 0.1740 - val_mae: 0.1740
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1729 - mae: 0.1729 - val_loss: 0.1707 - val_mae: 0.1707
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1685 - mae: 0.1685 - val_loss: 0.1674 - val_mae: 0.1674
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1666 - mae: 0.1666 - val_loss: 0.1661 - val_mae: 0.1661
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/

> Loading TensorBoard
- Install extension: Tensorboard
- Double Run 2nd block code else use port: as shows, ex: http://localhost:6006/?darkMode=true#timeseries
---


In [19]:
# %pip install tensorboard
# %load_ext tensorboard
%reload_ext tensorboard

In [20]:
# %tensorboard --logdir logs/fit
%tensorboard --logdir regressionlogs/fit

ERROR: Failed to launch TensorBoard (exited with 1).

>Evaluating Test Data
---

In [21]:
test_loss, test_mae = models.evaluate(X_test, y_test)
print(f'Test MAE: {test_mae}')

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1448 - mae: 0.1448
Test MAE: 0.1448485106229782


> Saving Model
---

In [22]:
models.save('models/reg-ann-customer-churn-prediction-model.h5')